In [106]:
import pandas as pd
from fetch_wallet import fetch_wallet_transactions, WALLET_ADDRESS

In [107]:
df = fetch_wallet_transactions(WALLET_ADDRESS)
print(df)

Found 19 transactions
                                                hash  \
0  0x3798896c921f2cf66b5f7bc567ec00c80f627ac9df0b...   
1  0x3b6a2a6c3323520b9122904e4da019612807db87f62e...   
2  0xc46afed57b15e53fc870a9dd928a12453bd29b8ac354...   
3  0x38e2c0e17555943ea356f383318a9d34150071ca4ab9...   
4  0x21d48c3e7324cff6503e8bf00f9b7ec7210c8a4672cb...   

                                         from  \
0  0x03fc89ca0408d337eb713aee85e8bca4ab1c3087   
1  0x03fc89ca0408d337eb713aee85e8bca4ab1c3087   
2  0x03fc89ca0408d337eb713aee85e8bca4ab1c3087   
3  0x03fc89ca0408d337eb713aee85e8bca4ab1c3087   
4  0x03fc89ca0408d337eb713aee85e8bca4ab1c3087   

                                           to              value   timeStamp  
0  0xcbeeb014be7ada9586f959790792df201fff59ed  37408619877165513  1774428400  
1  0x779ded0c9e1022225f8e0630b35a9b54be713736                  0  1773132884  
2  0x779ded0c9e1022225f8e0630b35a9b54be713736                  0  1772757128  
3  0x779ded0c9e1022225f8e0630b

In [108]:
df['timeStamp'] = pd.to_datetime(df['timeStamp'].astype(int), unit='s')
df['trade_hour'] = df['timeStamp'].dt.hour
df['day_of_week'] = df['timeStamp'].dt.day_name()

print(df[['trade_hour', 'day_of_week']])

    trade_hour day_of_week
0            8   Wednesday
1            8     Tuesday
2            0      Friday
3            2      Monday
4            9    Saturday
5            7    Saturday
6            6      Friday
7           13      Sunday
8            9      Sunday
9            3      Monday
10           8      Sunday
11           8    Thursday
12           9      Sunday
13           6      Sunday
14           7      Friday
15           7   Wednesday
16           3    Saturday
17           3    Thursday
18          10      Monday


In [109]:
df['day_of_week'].mode()[0]

'Sunday'

In [110]:
df['trade_hour'].mode()[0]

np.int32(8)

In [111]:
df['time_since_last_trade'] = df['timeStamp'].diff()
df['time_since_last_trade'] = df['time_since_last_trade'].dt.total_seconds() / 3600
df['time_since_last_trade'] = df['time_since_last_trade'].abs()
print(df[['time_since_last_trade']])

    time_since_last_trade
0                     NaN
1              359.865556
2              104.376667
3               93.822778
4             1385.581667
5              505.319444
6              193.510000
7              113.075556
8                3.971667
9              486.253333
10              18.934444
11              71.522778
12              94.688889
13               3.387778
14             382.523333
15              48.460000
16              99.674444
17              48.368889
18             401.141111


In [112]:
df['time_since_last_trade'].mean()

np.float64(245.2487962962963)

In [113]:
df['Value'] = df['value'].astype(float) / 1e18
df['value_zscore'] = (df['Value'] - df['Value'].mean()) / df['Value'].std()
print(df[['Value', 'value_zscore']])

       Value  value_zscore
0   0.037409      1.703813
1   0.000000     -0.372124
2   0.000000     -0.372124
3   0.000000     -0.372124
4   0.000000     -0.372124
5   0.020000      0.737747
6   0.000000     -0.372124
7   0.000000     -0.372124
8   0.000000     -0.372124
9   0.000000     -0.372124
10  0.000000     -0.372124
11  0.000000     -0.372124
12  0.000000     -0.372124
13  0.000000     -0.372124
14  0.000000     -0.372124
15  0.000000     -0.372124
16  0.000000     -0.372124
17  0.000000     -0.372124
18  0.070000      3.512425


In [114]:
df['value_zscore'].max()

3.512424859142834

In [115]:
date_counts = df['timeStamp'].dt.date.value_counts().sort_index()
df['trade_per_day'] = df['timeStamp'].dt.date.map(date_counts)
print(df[['timeStamp', 'trade_per_day']])

             timeStamp  trade_per_day
0  2026-03-25 08:46:40              1
1  2026-03-10 08:54:44              1
2  2026-03-06 00:32:08              1
3  2026-03-02 02:42:46              1
4  2026-01-03 09:07:52              1
5  2025-12-13 07:48:42              1
6  2025-12-05 06:18:06              1
7  2025-11-30 13:13:34              2
8  2025-11-30 09:15:16              2
9  2025-11-10 03:00:04              1
10 2025-11-09 08:04:00              1
11 2025-11-06 08:32:38              1
12 2025-11-02 09:51:18              2
13 2025-11-02 06:28:02              2
14 2025-10-17 07:56:38              1
15 2025-10-15 07:29:02              1
16 2025-10-11 03:48:34              1
17 2025-10-09 03:26:26              1
18 2025-09-22 10:17:58              1


In [116]:
df['trade_per_day'].max()

2

In [117]:
error_rate = (df['isError'] == '1').sum()/len(df)*100
print(f"Error Rate: {error_rate:.2f}%")

Error Rate: 0.00%


In [118]:
error_rate

np.float64(0.0)

In [119]:
print(df.columns)

Index(['blockNumber', 'timeStamp', 'hash', 'nonce', 'blockHash',
       'transactionIndex', 'from', 'to', 'value', 'gas', 'gasPrice', 'isError',
       'txreceipt_status', 'input', 'contractAddress', 'cumulativeGasUsed',
       'gasUsed', 'confirmations', 'methodId', 'functionName', 'L1Gas',
       'L1GasPrice', 'L1FeesPaid', 'L1FeeScalar', 'trade_hour', 'day_of_week',
       'time_since_last_trade', 'Value', 'value_zscore', 'trade_per_day'],
      dtype='object')


In [120]:
def simplify_defi_action(action):
    action = action.lower()
    if 'transfer' in action:
        return 'Token Transfer'
    elif 'swap' in action or 'exactinput' in action or 'exactoutput' in action:
        return 'Token Swap'
    elif 'addliquidity' in action or 'mint' in action:
        return 'Add Liquidity'
    elif 'removeliquidity' in action or 'burn' in action:
        return 'Remove Liquidity'
    elif 'deposit' in action:
        return 'Vault Deposit'
    elif 'withdraw' in action:
        return 'Vault Withdrawal'
    elif 'approve' in action:
        return 'Token Approval'
    else:
        return 'DeFi Interaction'

most_common_defi_action = simplify_defi_action(df['functionName'].mode()[0])
print(most_common_defi_action)

Token Transfer
